Mohammad Nayebi 

SVM in Polarization 

In [1]:
import cv2
import numpy as np
import os
import csv
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import datetime
import sunposition as sp
from pysolar.solar import get_azimuth
from zoneinfo import ZoneInfo


Data Loading

In [ ]:
metadata_csv_path = r"Path\to...\Simulation_Results\simulation_metadata.csv"
base_dir = os.path.dirname(metadata_csv_path)

iterations                = []
time_vec                  = []
Psi_true_deg_vec          = []
Psi_sn_deg_vec            = []
zen_deg_vec               = []
rotation_axis_azimuth_vec = []
rotation_angle_deg_vec    = []
data_file_vec             = []

AoP_local_list = []
DoP_list       = []

with open(metadata_csv_path, newline='') as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        iterations.append(int(row["iteration"]))
        time_vec.append(float(row["time"]))
        Psi_true_deg_vec.append(float(row["Psi_true_deg"]))
        Psi_sn_deg_vec.append(float(row["Psi_sn_deg"]))
        zen_deg_vec.append(float(row["zen_deg"]))
        rotation_axis_azimuth_vec.append(float(row["rotation_axis_azimuth_deg"]))
        rotation_angle_deg_vec.append(float(row["rotation_angle_deg"]))
        data_file_vec.append(row["data_file"])

        npz_path = os.path.join(base_dir, row["data_file"])
        npz_data = np.load(npz_path)

        AoP_local_list.append(npz_data["AoP_local"])
        DoP_list.append(npz_data["DoP"])

AOP_deg = np.degrees(np.array(AoP_local_list))
DOP = np.array(DoP_list)

print("Loaded:", len(AOP_deg), "frames")
print("AoP_local shape:", AOP_deg.shape)
print("DoP shape:", DOP.shape)

print("AOP range: [{:.3f}, {:.3f}] deg".format((AOP_deg).min(), (AOP_deg).max()))
print("DOP range: [{:.3f}, {:.3f}]    ".format((DOP).min(), (DOP).max()))

# plt.figure(figsize=(6, 6))
# plt.imshow(AOP_deg[3], cmap='jet', vmin=-90, vmax=90)
# plt.colorbar(label='AoLP (deg)')
# plt.axis('off')
# plt.title('Angle of Polarization (AoLP, meridian frame)')
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(6, 6))
# plt.imshow(DOP[3], cmap='jet', vmin=-1, vmax=1)
# plt.colorbar(label='AoLP (deg)')
# plt.axis('off')
# plt.title('Angle of Polarization (AoLP, meridian frame)')
# plt.tight_layout()
# plt.show()


Loaded: 300 frames
AoP_local shape: (300, 1024, 1224)
DoP shape: (300, 1024, 1224)
AOP range: [-90.000, 90.000] deg
DOP range: [0.000, 0.437]    


Parameters

In [3]:
T = 30 # degrees
r0_list = [] # |AoP| >= T
r1_list = [] # |AoP| < T

r2_list = [] # largest component
r3_list = [] # second largest component

step3_list = []

MAX_ITER        = 10  # maximum iterations (fallback if convergence not reached)
N_SAMPLES       = 200 # number of samples per iteration (balanced: N/2 each class)
SHIFT_DEG       = 3   # angular band half-width around current line (degrees)
C_SVM           = 1.0 # SVM regularization parameter
CONVERGENCE_TOL = 0.2 # stopping criterion: angle change threshold (degrees)
theta_mes_vec = []     # final symmetry angle per frame

yaw_mes_vec1 = []
yaw_mes_vec2 = []

yaw measurement loop

In [4]:
for iter in range(len(AOP_deg)):
        
    # ----- Step 1: Threshold -----
    
    AOP_frame = AOP_deg[iter]
    r0 = np.abs(AOP_frame) >= T
    r1 = np.abs(AOP_frame) < T
    r0_list.append(r0)
    r1_list.append(r1)

    # ----- Step 2: r0 --> r2 and r3 -----
    
    r0_uint8 = (r0.astype(np.uint8) * 255)
    kernel = np.ones((3, 3), np.uint8)
    r0_eroded = cv2.erode(r0_uint8, kernel, iterations=1)
    num_labels, labels = cv2.connectedComponents(r0_eroded)
    if num_labels < 3:
        raise ValueError(
            f"Frame {iter}: Yaw estimation failed. "
            f"Expected at least 2 components, but found {num_labels - 1}."
        )
    areas = [(labels == i).sum() for i in range(1, num_labels)]
    order = np.argsort(areas)[::-1]
    r2 = labels == (order[0] + 1)
    r3 = labels == (order[1] + 1)
    r2_list.append(r2)
    r3_list.append(r3)

    # ----- Step 3: Antisym --> Sym -----
    
    AOP_step3_deg = np.zeros_like(AOP_frame)
    AOP_step3_deg[r2] = -AOP_frame[r2]
    AOP_step3_deg[r3] =  AOP_frame[r3]
    step3_list.append(AOP_step3_deg)
    
    # ----- SVM method -----
        
    h, w = AOP_step3_deg.shape
    iy, ix = np.indices((h, w))
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    
    labels_map = np.zeros_like(AOP_step3_deg, dtype=np.int32)
    labels_map[AOP_step3_deg > 0] =  1
    labels_map[AOP_step3_deg < 0] = -1
    
    X_all_raw = np.column_stack((ix.ravel(), iy.ravel())).astype(np.float64)
    X_all_raw[:, 0] -= cx
    X_all_raw[:, 1] -= cy
    
    y_all = labels_map.ravel()

    valid_mask = y_all != 0
    X_valid = X_all_raw[valid_mask]
    y_valid = y_all[valid_mask]

    idx_pos = np.where(y_valid ==  1)[0]
    idx_neg = np.where(y_valid == -1)[0]

    N_half = N_SAMPLES // 2

    theta_history = []  
    current_theta_deg = None  

    # Sampling
    for i in range(MAX_ITER):
    
        if i == 0:
            sel_pos = np.random.choice(idx_pos, N_half, replace=False)
            sel_neg = np.random.choice(idx_neg, N_half, replace=False)

        else:
            phi_all = np.degrees(np.arctan2(X_valid[:, 1], X_valid[:, 0]))
            delta = (phi_all - current_theta_deg) % 180.0
            delta = np.where(delta > 90.0, delta - 180.0, delta)
            near_line = np.abs(delta) < SHIFT_DEG
            band_pos = np.intersect1d(idx_pos, np.where(near_line)[0])
            band_neg = np.intersect1d(idx_neg, np.where(near_line)[0])
            n_pos = min(N_half, len(band_pos)) if len(band_pos) >= 50 else N_half
            n_neg = min(N_half, len(band_neg)) if len(band_neg) >= 50 else N_half
            pool_pos = band_pos if len(band_pos) >= 50 else idx_pos
            pool_neg = band_neg if len(band_neg) >= 50 else idx_neg
            sel_pos = np.random.choice(pool_pos, n_pos, replace=False)
            sel_neg = np.random.choice(pool_neg, n_neg, replace=False)
    
        # Fit SVM 
        idx_sel = np.concatenate([sel_pos, sel_neg])
        np.random.shuffle(idx_sel)
        X_sub = X_valid[idx_sel]
        y_sub = y_valid[idx_sel]
        svm = SVC(kernel='linear', C=C_SVM)
        svm.fit(X_sub, y_sub)
        w_vec = svm.coef_[0]
        b_val = svm.intercept_[0]
        theta_rad = np.arctan2(-w_vec[0], w_vec[1])
        theta_deg = np.degrees(theta_rad)
        theta_history.append(theta_deg)
        current_theta_deg = theta_deg

    # Final angle
    theta_final_deg = np.mean(theta_history[2:]) 
    theta_final_rad = np.radians(theta_final_deg)
    theta_mes_vec.append(theta_final_deg)
    
    # ----- yaw measurement
    Psi_sc = (180 + theta_deg) % 360
    Psi_sc = Psi_sc % 180
    Psi_raw = (-Psi_sn_deg_vec[iter] + Psi_sc) % 360
    Psi_est_1 = Psi_raw
    Psi_est_2 = (Psi_raw + 180) % 360
    print(f"Yaw candidates (180° ambiguity) at frame {iter}:")
    print(Psi_est_1, "degrees")
    print(Psi_est_2, "degrees")
    yaw_mes_vec1.append(Psi_est_1)
    yaw_mes_vec2.append(Psi_est_2)


Yaw candidates (180° ambiguity) at frame 0:
0.10608573799902388 degrees
180.10608573799902 degrees
Yaw candidates (180° ambiguity) at frame 1:
359.8053829921425 degrees
179.80538299214254 degrees
Yaw candidates (180° ambiguity) at frame 2:
0.2785466233351457 degrees
180.27854662333516 degrees
Yaw candidates (180° ambiguity) at frame 3:
359.869188025409 degrees
179.86918802540902 degrees
Yaw candidates (180° ambiguity) at frame 4:
0.0043858394432589876 degrees
180.00438583944327 degrees
Yaw candidates (180° ambiguity) at frame 5:
359.7088882763011 degrees
179.70888827630108 degrees
Yaw candidates (180° ambiguity) at frame 6:
359.9616267025824 degrees
179.9616267025824 degrees
Yaw candidates (180° ambiguity) at frame 7:
359.9355596901096 degrees
179.93555969010959 degrees
Yaw candidates (180° ambiguity) at frame 8:
0.27330918934598003 degrees
180.27330918934598 degrees
Yaw candidates (180° ambiguity) at frame 9:
359.95848956659995 degrees
179.95848956659995 degrees
Yaw candidates (180° a

Save results to csv

In [5]:
# Create "Result" folder in current directory if not exists
result_dir = os.path.join(os.getcwd(), "Result")
os.makedirs(result_dir, exist_ok=True)

# Output CSV path
output_csv_path = os.path.join(result_dir, "yaw_results.csv")

# Write CSV
with open(output_csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)    
    # Header
    writer.writerow(["iteration", "time", "Psi_true_deg", "yaw_mes_1", "yaw_mes_2"])    
    # Rows
    for i in range(len(iterations)):
        writer.writerow([
            iterations[i],
            time_vec[i],
            Psi_true_deg_vec[i],
            yaw_mes_vec1[i],
            yaw_mes_vec2[i]
        ])
print("Saved results to:", output_csv_path)


Saved results to: d:\University\BSC\Projects\Polarization\M.Nayebi\SVM\Codes\Simulation\Result\yaw_results.csv
